# Glider / CTD Map App — Barkley Sound

Interactive Leaflet basemap for browsing glider tracks and CTD casts spatially, instead of
scrolling through notebook cells looking for the right dataset. Every marker/line on the map is
clickable and pops up the *same* plot `Glider_Curtain_Plot.ipynb` would produce for that dataset
(reused, not reimplemented, from `glider_lib.py` — see that file's docstring for how the two
notebooks stay in sync).

- **Glider tracks** → one polyline per *deployment*, drawn from the real C-PROOF archives in
  `data/` (`cproof_glider_realtime.nc`, `cproof_glider_delayed.nc`). Click one → 3D temperature
  curtain popup for that deployment (`plot_glider_curtain`).
- **CTD casts** → a single marker at the cast's (lon, lat). Click it → 2D profile popup
  (`plot_ctd_profile`). Skipped if the cast file isn't present.
- **Basemap** → Esri Ocean Basemap, framed to the region's bounding box from
  `Glider_Curtain_Plot.ipynb`'s `CONFIG["REGION"]`.

Real-time and delayed-mode tracks are separate, independently toggleable layers, because the
difference between them is scientific rather than cosmetic: real-time is uncalibrated and
decimated for satellite bandwidth, delayed-mode is calibrated, quality controlled, and hundreds
of times denser. See `data/README.md`.

This uses **ipyleaflet** (a real Leaflet.js map wired to live Python callbacks via Jupyter
widgets), not a static HTML map — clicking a layer re-runs a plotting function against that
layer's data right now, in this kernel. See the last section for how to launch this outside of
JupyterLab as a standalone app (Voila).

**Roadmap** (not built yet, noted here so the structure below stays extensible):
- The other six science variables the archives carry (salinity, density, oxygen, chlorophyll,
  backscatter, CDOM) as a variable selector — this notebook reads temperature only, since it is
  the one variable present on every deployment.
- Long-term mooring time series as another clickable layer type.
- Full-grid netCDF current fields (e.g. modeled u/v velocity) as a vector-field overlay on the
  same map — planned as either periodic quiver/streamline raster tiles or a dedicated Leaflet
  velocity plugin, added as one more `ipyleaflet.Layer` alongside the marker/polyline layers
  below, so it composes with everything already here rather than needing a separate map.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from ipyleaflet import (
    Map, Marker, Polyline, Popup, basemaps,
    LayersControl, LayerGroup, ScaleControl, FullScreenControl,
)
from ipywidgets import Output

from glider_lib import (
    GLIDER_VARIABLE,
    GLIDER_VARIABLE_LABEL,
    load_platform_data,
    load_glider_archive,
    glider_tracks,
    track_vertices,
    decimate,
    plot_ctd_profile,
    plot_glider_curtain,
)

## 2. Configuration

Mirrors the relevant parts of `Glider_Curtain_Plot.ipynb`'s `CONFIG` cell — same `REGION`
bounding box and CTD dataset path/column map. Kept as a separate dict here (rather than imported
from the other notebook) since `.ipynb` files aren't directly importable; if `CONFIG` in the
curtain notebook changes, update it here too.

The `GLIDER` block no longer carries a `DATA_PATH`/`COLUMN_MAP`. Those described a single-track
file; the real C-PROOF archives hold many deployments stacked on one dimension, and
`glider_lib.load_glider_archive()` already knows how to read that layout. What's configurable
here is which archives to show, how far back, and how the two are styled.

In [ ]:
CONFIG_MAP = {
    "REGION": {"lon_range": (-126.8, -124.5), "lat_range": (47.85, 49.36)},  # Barkley Sound, BC

    "CTD": {
        "DATA_PATH": "NE_San_Diego_Trough_Aug_2022.csv",
        "FILE_TYPE": "csv",
        "COLUMN_MAP": {"lon": None, "lat": None, "depth": None, "variable": "Salt2"},
        "VARIABLE_LABEL": "Salinity (PSU)",
        "LINE_COLOR": "#1b6ca8",
        "DEPTH_POSITIVE_DOWN": True,
        "MARKER_COLOR": "#d1495b",
    },

    # No DATA_PATH/COLUMN_MAP here any more: the glider layer reads the real C-PROOF
    # archives in data/ through glider_lib.load_glider_archive(), which knows how they
    # are laid out (many deployments stacked on one `obs` dimension). Temperature only --
    # it is the one variable flown on every deployment.
    "GLIDER": {
        "MODES": ["realtime", "delayed"],  # drop "delayed" for a fast-loading map
        "LAST_DAYS": None,                 # None = whole archive; 7 for a "this week" view
        "VARIABLE_LABEL": GLIDER_VARIABLE_LABEL,
        "COLOR_SCALE": "Thermal",
        "DEPTH_POSITIVE_DOWN": True,
        "MAX_POPUP_POINTS": 4000,          # decimation target for a popup curtain
        "STYLE": {
            # Two visually distinct layers, because the difference is scientific, not
            # cosmetic: real-time is uncalibrated and heavily decimated for satellite
            # bandwidth; delayed-mode is calibrated, quality controlled, and far denser.
            "realtime": {"LINE_COLOR": "#f4a261", "WEIGHT": 3, "LABEL": "Glider tracks (real-time)"},
            "delayed":  {"LINE_COLOR": "#2a9d8f", "WEIGHT": 2, "LABEL": "Glider tracks (delayed-mode)"},
        },
    },
}

# Popup plots are rendered small so they read as a preview, not a full dashboard panel.
POPUP_FIGURE_SIZE = dict(width=380, height=320)

## 3. Load CTD + glider data

The glider layer reads the **real C-PROOF archives** in `data/`, via
`glider_lib.load_glider_archive()` — no more synthetic sample track. Each archive stacks many
deployments on one `obs` dimension, so this cell loads a frame per mode and splits it into
per-deployment tracks.

- **`realtime`** — committed to git (~3 MB), updated nightly, 27 deployments spanning 2022–2026.
  Uncalibrated and heavily decimated for satellite bandwidth.
- **`delayed`** — calibrated and quality controlled, ~650× denser, 10 deployments. **Gitignored**
  because it is ~37 MB, so on a fresh clone it won't exist; this cell reports that and carries on
  with real-time only. Rebuild it with `python data/update_cproof_glider.py --mode delayed`.

Temperature only — it is the one variable flown on every deployment (oxygen is missing on
roughly a third), and reading one variable rather than all seven is what keeps the delayed
archive's three million observations manageable in memory. Expect this cell to take a few
seconds and a few hundred MB when `delayed` is included; drop it from `MODES` for a fast map.

The CTD cast is loaded the same way it always was, but its file is not in this repo — the cell
now skips that layer rather than raising if it's absent.

In [ ]:
ctd_cfg = CONFIG_MAP["CTD"]
ctd_var = ctd_cfg["COLUMN_MAP"]["variable"]
try:
    ctd_df = load_platform_data(ctd_cfg["DATA_PATH"], ctd_cfg["FILE_TYPE"], ctd_cfg["COLUMN_MAP"])
    if ctd_cfg["DEPTH_POSITIVE_DOWN"]:
        ctd_df["Depth"] = -ctd_df["Depth"].abs()
    print(f"CTD cast: {len(ctd_df)} rows at "
          f"({ctd_df['Longitude'].iloc[0]:.3f}, {ctd_df['Latitude'].iloc[0]:.3f})")
except FileNotFoundError:
    ctd_df = None
    print(f"CTD: '{ctd_cfg['DATA_PATH']}' not found -- skipping the CTD marker. "
          "(That file is a San Diego Trough cast and never was in this repo; it sits well "
          "outside the Barkley Sound region anyway. Point CONFIG_MAP['CTD']['DATA_PATH'] at "
          "a local cast to bring the layer back.)")

# --- Glider: the real C-PROOF archives -------------------------------------------
# One frame per mode, each split into per-deployment tracks. The delayed archive is
# gitignored (~37 MB) and rebuilt with `python data/update_cproof_glider.py --mode
# delayed`, so its absence is reported and skipped rather than raised.
glider_cfg = CONFIG_MAP["GLIDER"]
glider_var = GLIDER_VARIABLE
glider_layers = {}

for mode in glider_cfg["MODES"]:
    try:
        frame = load_glider_archive(mode, last_days=glider_cfg["LAST_DAYS"])
    except FileNotFoundError as missing:
        print(f"Glider ({mode}): {missing}")
        continue

    if frame.empty:
        print(f"Glider ({mode}): archive holds no observations for this window -- skipping.")
        continue

    if glider_cfg["DEPTH_POSITIVE_DOWN"]:
        frame["Depth"] = -frame["Depth"].abs()

    tracks = glider_tracks(frame)
    glider_layers[mode] = tracks
    span = f"{frame['time'].min():%Y-%m-%d} to {frame['time'].max():%Y-%m-%d}"
    print(f"Glider ({mode}): {len(frame):,} observations, {len(tracks)} deployments, {span}")

if not glider_layers:
    raise RuntimeError(
        "No glider archive could be loaded. Build one with:\n"
        "  python data/update_cproof_glider.py --mode realtime"
    )

# Sanity check against REGION -- a mismatch here just means the initial map view won't be
# centered on this dataset (you can still pan/zoom to it); it doesn't block anything below.
lon_lo, lon_hi = CONFIG_MAP["REGION"]["lon_range"]
lat_lo, lat_hi = CONFIG_MAP["REGION"]["lat_range"]
checks = [("CTD", ctd_df)] if ctd_df is not None else []
checks += [(f"glider/{mode}", pd.concat(tracks.values()))
           for mode, tracks in glider_layers.items()]
for name, df in checks:
    out_of_region = not ((lon_lo <= df["Longitude"]).all() and (df["Longitude"] <= lon_hi).all()
                          and (lat_lo <= df["Latitude"]).all() and (df["Latitude"] <= lat_hi).all())
    if out_of_region:
        print(f"  ⚠ {name} data falls outside CONFIG_MAP['REGION']'s bounding box "
              f"(lon [{lon_lo}, {lon_hi}], lat [{lat_lo}, {lat_hi}]) -- the map will still frame "
              "REGION on load, but you'll need to pan to see this layer.")

## 4. Build the basemap

Framed to `CONFIG_MAP["REGION"]` via `fit_bounds`, exactly as requested — the initial view is
the region's bounding box, independent of where any loaded dataset actually falls (see the
warning above if they don't overlap).

In [9]:
m = Map(
    basemap=basemaps.Esri.OceanBasemap,
    center=((lat_lo + lat_hi) / 2, (lon_lo + lon_hi) / 2),
    zoom=8,
    scroll_wheel_zoom=True,
)
m.fit_bounds([[lat_lo, lon_lo], [lat_hi, lon_hi]])
m.add(LayersControl(position="topright"))
m.add(ScaleControl(position="bottomleft"))
m.add(FullScreenControl())
m

Map(center=[48.605000000000004, -125.65], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_…

## 5. Click-to-plot layers

Each layer's `on_click` handler builds the popup lazily, the first time that layer is clicked
(not up front for every dataset) — cheap for a couple of layers now, and keeps this cheap once
there are many.

In [ ]:
def make_popup_handler(build_figure, location):
    '''Returns an on_click handler that lazily builds a Plotly-in-Popup the first time it fires.'''
    state = {"popup": None}

    def handler(**kwargs):
        if state["popup"] is None:
            fig = build_figure()
            fig.update_layout(**POPUP_FIGURE_SIZE, margin=dict(l=40, r=10, t=40, b=40))
            output = Output()
            with output:
                from IPython.display import display
                display(fig)
            state["popup"] = Popup(location=location, child=output, close_button=True, auto_close=False)
        if state["popup"] not in m.layers:
            m.add(state["popup"])

    return handler


# --- CTD marker ---
if ctd_df is not None:
    ctd_location = (ctd_df["Latitude"].iloc[0], ctd_df["Longitude"].iloc[0])
    ctd_marker = Marker(location=ctd_location, title="CTD cast", draggable=False)
    ctd_marker.on_click(make_popup_handler(
        lambda: plot_ctd_profile(ctd_df, ctd_var, variable_label=ctd_cfg["VARIABLE_LABEL"],
                                  line_color=ctd_cfg["LINE_COLOR"]),
        ctd_location,
    ))
    m.add(ctd_marker)

# --- Glider tracks: one polyline per deployment ---
# Not one polyline over the whole archive: it stacks 27 real-time deployments spanning
# 2022-2026, so a single line would run straight from the end of one deployment to the
# start of the next. Each mode becomes a LayerGroup so the layers control can toggle
# real-time and delayed independently.
def build_curtain(track, deployment):
    '''Curtain plot for one deployment, decimated to stay interactive in a popup.'''
    thinned = decimate(track, glider_cfg["MAX_POPUP_POINTS"])
    return plot_glider_curtain(
        thinned, glider_var,
        variable_label=glider_cfg["VARIABLE_LABEL"],
        color_scale=glider_cfg["COLOR_SCALE"],
        title=f"{deployment}<br><sub>{len(track):,} obs, showing {len(thinned):,}</sub>",
    )


for mode, tracks in glider_layers.items():
    style = glider_cfg["STYLE"][mode]
    lines = []
    for deployment, track in tracks.items():
        vertices = track_vertices(track)
        if len(vertices) < 2:
            continue  # a single surfacing isn't a track -- nothing to draw
        line = Polyline(locations=vertices, color=style["LINE_COLOR"],
                        weight=style["WEIGHT"], fill=False)
        line.on_click(make_popup_handler(
            lambda track=track, deployment=deployment: build_curtain(track, deployment),
            vertices[len(vertices) // 2],
        ))
        lines.append(line)

    m.add(LayerGroup(layers=tuple(lines), name=style["LABEL"]))
    print(f"{style['LABEL']}: {len(lines)} deployment tracks")

print("\nClick any glider track on the map above to pop up that deployment's curtain plot.")

## 6. Launching this as a standalone app

Right now this runs inline in JupyterLab, which is enough to use it interactively today. To
launch it as its own browser tab without the notebook/code chrome (a real "app"), serve it with
[Voila](https://voila.readthedocs.io/) — it re-executes this notebook and shows only the
rendered widgets, with the same live click → Python → popup behavior:

```bash
pip install voila   # not installed in this environment yet
voila Glider_Map_App.ipynb
```

On this JupyterHub, `jupyter-server-proxy` is already installed, so a running Voila server is
reachable at `<hub-url>/user/<your-username>/voila/render/final_notebooks/Glider_Map_App.ipynb`
without opening any extra ports — that URL is shareable with anyone who can reach the hub.